# RPC Laborator 5

Implementati jocul Connect 4 cu Minimax si Alpha-Beta pruning - reguli gasiti aici https://www.buffalolib.org/sites/default/files/gaming-unplugged/inst/Connect%204%20Instructions.pdf

Mare atentie la reguli - nu putem plasa piesa oriunde, ci o punem pe o coloana la alegere si va "cadea" pana la ulimul loc liber.

Dupa ce ati implementat varianta "AI vs AI", inlocuiti nodurile MIN cu user input(jucati voi vs algo)

In [61]:
from copy import deepcopy

MAX=1
MIN=2
GOL=0
WIN=10000000

class Nod:
    #as fi numit-o mai degraba stare ca e mai "fitting" dar asa am facut pana acum asa ramane
    def __init__(self, info, parinte=None, detalii_mutare=None):
        self.parinte = parinte
        self.info = info
        self.detalii=detalii_mutare


    def evaluare(self):
         #aici implementam functia de evaluare
        scor=0
        matrice=self.info # tabla de joc


        #calculam scorul pe linii
        for i in range(6):
            linie=matrice[i]
            for j in range(4):
                lista=linie[j:j+4]
                scor+=self.calculeaza_scor(lista)

        #calculam scorul pe coloana
        for j in range(7):
            coloana=[matrice[i][j] for i in range(6)]
            for i in range(3):
                lista=coloana[i:i+4]
                scor+=self.calculeaza_scor(lista)

        #calucalm pe diagonala principala
        for i in range(3):
            for j in range(4):
                lista= [matrice[i+k][j+k] for k in range(4)]
                scor+=self.calculeaza_scor(lista)

        #diag secundara
        for i in range(3):
            for j in range(3, 7):
                lista = [matrice[i+k][j-k] for k in range(4)]
                scor += self.calculeaza_scor(lista)

        return scor




    def calculeaza_scor(self, lista):

        #daca incep in mijloc am 7 posbilitati de castiga
        #nr piese eu, nr piese avs, nr gaoel



        #trebuie sa fie simetrice
        puncte=0
        if lista.count(MAX)==4:
            puncte+=WIN
        elif lista.count(MAX)==3 and lista.count(GOL)==1:
            puncte+=100
        elif lista.count(MAX)==2 and lista.count(GOL)==2:
            puncte+=10

        if lista.count(MIN)==4:
            puncte-=WIN
        elif lista.count(MIN)==3 and lista.count(GOL)==1:
            puncte-=100
        elif lista.count(MIN)==2 and lista.count(GOL)==2:
            puncte-=10


        return puncte


    def __str__(self):
        matrice=self.info
        rezultat=""
        for i in range(5,-1,-1):
            linie=""
            for j in range(7):
                linie=linie+str(matrice[i][j])+" "
            rezultat+=linie+"\n"
        return rezultat



    def __repr__(self):
        return self.__str__()

class Tree:
    def __init__(self, nod_start=None):
        if nod_start is not None:
            self.nod_start=nod_start
        else:
            self.nod_start=Nod([[0 for _ in range(7)] for _ in range(6)],parinte=None, detalii_mutare=None)




    def este_final(self, nod): #true daca e stare finala. eu personal as returna si castigatorul/draw (intre 0,1,2,3 de ex, 0 draw, 1 castiga AI 2 castiga om, 3 nu e final). va ia din munca mai incolo

        matrice=nod.info
        scor=nod.evaluare()
        if scor>=WIN:
            return True, MAX
        elif scor<=-WIN:
            return True, MIN


        #remiza
        if GOL not in matrice[5]:
            return True, 0
        else:
            return False, 3





    def succesori(self, nod, jucator):
#fill in the blanks unde e mai sus, posibil sa nu aveti nevoie si de str si de repr. atata timp cat se afiseaza lizibil starile prin care trecem e ok
        lista_succesori=[]
        matrice=nod.info

        for j in range(7):
            #verificam ca coloana sa nu fie plina
            if matrice[5][j] == GOL:
                matrice_noua = deepcopy(matrice)
                #incepem de jos si cautam in sus
                for i in range(6):
                    if matrice_noua[i][j] == GOL:
                        matrice_noua[i][j] = jucator
                        break

                lista_succesori.append(Nod(matrice_noua, parinte=nod))

        return lista_succesori






#ramane de implementat minimaxu

### MIN MAX

In [62]:
def minimax(nod, adancime, jucator, tree):

    final,castigator=tree.este_final(nod=nod)

    if adancime==0 or final:
        return nod.evaluare(), nod

    if jucator==MAX:
        maxi=-WIN*5
        maxiNod = None
        succesori=tree.succesori(nod, jucator=MAX)

        for succesor in succesori:
            evaluare,_ =minimax(succesor, adancime-1, MIN, tree)
            if evaluare>maxi:
                maxi=evaluare
                maxiNod=succesor
        return maxi, maxiNod

    else:
        mini=5*WIN
        miniNod=None
        succesori=tree.succesori(nod, jucator=MIN)

        for succesor in succesori:
            evaluare,_=minimax(succesor, adancime-1, MAX, tree)
            if evaluare<mini:
                mini=evaluare
                miniNod=succesor
        return mini, miniNod









In [63]:
table=Tree()
print(minimax(table.nod_start, 0,MAX,table))


(0, 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 
0 0 0 0 0 0 0 
0 0 0 0 0 0 0 
0 0 0 0 0 0 0 
0 0 0 0 0 0 0 
)


In [64]:
def alpha_beta(nod, adancime, jucator, tree, alpha, beta):

    final,castigator=tree.este_final(nod=nod)

    if adancime==0 or final:
        return nod.evaluare(), nod

    if jucator==MAX:
        maxi=-WIN*5
        maxiNod = None
        succesori=tree.succesori(nod, jucator=MAX)

        for succesor in succesori:
            evaluare,_ =alpha_beta(succesor, adancime-1, MIN, tree, alpha, beta)
            if evaluare>maxi:
                maxi=evaluare
                maxiNod=succesor

            alpha = max(alpha, maxi)
            if beta <= alpha:
                break  # Alpha-Beta Pruning: tăiem ramura

        return maxi, maxiNod

    else:
        mini=5*WIN
        miniNod=None
        succesori=tree.succesori(nod, jucator=MIN)

        for succesor in succesori:
            evaluare,_=alpha_beta(succesor, adancime-1, MAX, tree, alpha, beta)
            if evaluare<mini:
                mini=evaluare
                miniNod=succesor

            beta = min(beta, mini)
            if beta <= alpha:
                break  # Alpha-Beta Pruning: tăiem ramura

        return mini, miniNod


### Game Loop


In [65]:
def gameloop():
    table = Tree()
    jucator = MAX

    while not table.este_final(table.nod_start)[0]:
        print(table.nod_start)
        print("----------------")


        _, table.nod_start = alpha_beta(table.nod_start, 3, jucator, table, -WIN*5, WIN*5)

       #schimbam tura
        jucator = MIN if jucator == MAX else MAX

    # Cand bucla se opreste, jocul s-a terminat
    print(table.nod_start)
    _, castigator = table.este_final(table.nod_start)
    print(f"Joc terminat! Castigator: {castigator}")

In [66]:
gameloop()

0 0 0 0 0 0 0 
0 0 0 0 0 0 0 
0 0 0 0 0 0 0 
0 0 0 0 0 0 0 
0 0 0 0 0 0 0 
0 0 0 0 0 0 0 

----------------
0 0 0 0 0 0 0 
0 0 0 0 0 0 0 
0 0 0 0 0 0 0 
0 0 0 0 0 0 0 
0 0 0 0 0 0 0 
1 0 0 0 0 0 0 

----------------
0 0 0 0 0 0 0 
0 0 0 0 0 0 0 
0 0 0 0 0 0 0 
0 0 0 0 0 0 0 
0 0 0 0 0 0 0 
1 0 2 0 0 0 0 

----------------
0 0 0 0 0 0 0 
0 0 0 0 0 0 0 
0 0 0 0 0 0 0 
0 0 0 0 0 0 0 
1 0 0 0 0 0 0 
1 0 2 0 0 0 0 

----------------
0 0 0 0 0 0 0 
0 0 0 0 0 0 0 
0 0 0 0 0 0 0 
2 0 0 0 0 0 0 
1 0 0 0 0 0 0 
1 0 2 0 0 0 0 

----------------
0 0 0 0 0 0 0 
0 0 0 0 0 0 0 
0 0 0 0 0 0 0 
2 0 0 0 0 0 0 
1 0 0 0 0 0 0 
1 1 2 0 0 0 0 

----------------
0 0 0 0 0 0 0 
0 0 0 0 0 0 0 
2 0 0 0 0 0 0 
2 0 0 0 0 0 0 
1 0 0 0 0 0 0 
1 1 2 0 0 0 0 

----------------
0 0 0 0 0 0 0 
0 0 0 0 0 0 0 
2 0 0 0 0 0 0 
2 0 0 0 0 0 0 
1 1 0 0 0 0 0 
1 1 2 0 0 0 0 

----------------
0 0 0 0 0 0 0 
0 0 0 0 0 0 0 
2 0 0 0 0 0 0 
2 0 0 0 0 0 0 
1 1 0 0 0 0 0 
1 1 2 2 0 0 0 

----------------
0 0 0 0 0 0 0 
0 0 0 0 0 0 0